In [1]:
###STEP 1 

In [2]:
#1
import mysql.connector
import pandas as pd
from getpass import getpass

In [3]:
#2
password = getpass("Enter your MySQL password: ")

connection = mysql.connector.connect(
    host='127.0.0.1',
    port=3306,
    user='root',
    password=password,
    database='sakila'
)

cursor = connection.cursor()

print("Connection successful!")

Enter your MySQL password:  ········


Connection successful!


In [4]:
# 3 
pd.read_sql("SHOW TABLES;", connection)

C:\Users\manda\AppData\Local\Temp\ipykernel_27804\4283134436.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql("SHOW TABLES;", connection)


,Tables_in_sakila
0,actor
1,actor_info
2,address
3,category
4,city
5,country
6,customer
7,customer_list
8,customer_rental_summary
9,film


In [5]:
###STEP 2

In [6]:
#1 Function to retrieve rentals for a specific month and year

def rentals_month(engine, month, year):
    query = f"""
    SELECT *
    FROM rental
    WHERE MONTH(rental_date) = {month}
    AND YEAR(rental_date) = {year};
    """
    rentals_df = pd.read_sql(query, engine)
    return rentals_df

In [7]:
#2 Retrieve May 2005 rentals
may_rentals = rentals_month(connection, 5, 2005)
may_rentals.head()

C:\Users\manda\AppData\Local\Temp\ipykernel_27804\1474216370.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  rentals_df = pd.read_sql(query, engine)


,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,1,2005-05-24 22:53:30,367,130,2005-05-26 22:04:30,1,2006-02-15 21:30:53
1,2,2005-05-24 22:54:33,1525,459,2005-05-28 19:40:33,1,2006-02-15 21:30:53
2,3,2005-05-24 23:03:39,1711,408,2005-06-01 22:12:39,1,2006-02-15 21:30:53
3,4,2005-05-24 23:04:41,2452,333,2005-06-03 01:43:41,2,2006-02-15 21:30:53
4,5,2005-05-24 23:05:21,2079,222,2005-06-02 04:33:21,1,2006-02-15 21:30:53


In [8]:
# 3 Retrieve June 2005 rentals
june_rentals = rentals_month(connection, 6, 2005)
june_rentals.head()

C:\Users\manda\AppData\Local\Temp\ipykernel_27804\1474216370.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  rentals_df = pd.read_sql(query, engine)


,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,1158,2005-06-14 22:53:33,1632,416,2005-06-18 21:37:33,2,2006-02-15 21:30:53
1,1159,2005-06-14 22:55:13,4395,516,2005-06-17 02:11:13,1,2006-02-15 21:30:53
2,1160,2005-06-14 23:00:34,2795,239,2005-06-18 01:58:34,2,2006-02-15 21:30:53
3,1161,2005-06-14 23:07:08,1690,285,2005-06-21 17:12:08,1,2006-02-15 21:30:53
4,1162,2005-06-14 23:09:38,987,310,2005-06-23 22:00:38,1,2006-02-15 21:30:53


In [9]:
#4 Check the shape of both DataFrames
print("May rentals:", may_rentals.shape)
print("June rentals:", june_rentals.shape)

May rentals: (1156, 7)
June rentals: (2311, 7)


In [10]:
#5 Count rentals per customer in May
may_customer_activity = (
    may_rentals
    .groupby("customer_id")
    .size()
    .reset_index(name="may_rentals")
)
may_customer_activity.head()

,customer_id,may_rentals
0,1,2
1,2,1
2,3,2
3,5,3
4,6,3


In [11]:
#6 Count rentals per customer in June

june_customer_activity = (
    june_rentals
    .groupby("customer_id")
    .size()
    .reset_index(name="june_rentals")
)
june_customer_activity.head()

,customer_id,june_rentals
0,1,7
1,2,1
2,3,4
3,4,6
4,5,5


In [12]:
#7 Find customers active in both May and June

active_both_months = pd.merge(
    may_customer_activity,
    june_customer_activity,
    on="customer_id",
    how="inner"
)
active_both_months.head()

,customer_id,may_rentals,june_rentals
0,1,2,7
1,2,1,1
2,3,2,4
3,5,3,5
4,6,3,4


In [13]:
#8 Compare customer activity between June and May

active_both_months["difference"] = (
    active_both_months["june_rentals"] 
    - active_both_months["may_rentals"]
)

active_both_months.head()

,customer_id,may_rentals,june_rentals,difference
0,1,2,7,5
1,2,1,1,0
2,3,2,4,2
3,5,3,5,2
4,6,3,4,1


In [14]:
#9 Add customer names

customer_df = pd.read_sql(
    """
    SELECT 
        customer_id,
        first_name,
        last_name
    FROM customer;
    """,
    connection
)

final_df = pd.merge(
    active_both_months,
    customer_df,
    on="customer_id",
    how="left"
)

final_df.head()

C:\Users\manda\AppData\Local\Temp\ipykernel_27804\1945799371.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  customer_df = pd.read_sql(


,customer_id,may_rentals,june_rentals,difference,first_name,last_name
0,1,2,7,5,MARY,SMITH
1,2,1,1,0,PATRICIA,JOHNSON
2,3,2,4,2,LINDA,WILLIAMS
3,5,3,5,2,ELIZABETH,BROWN
4,6,3,4,1,JENNIFER,DAVIS


In [15]:
#10 Reorder columns

final_df = final_df[
    [
        "customer_id",
        "first_name",
        "last_name",
        "may_rentals",
        "june_rentals",
        "difference"
    ]
]

final_df.head()

,customer_id,first_name,last_name,may_rentals,june_rentals,difference
0,1,MARY,SMITH,2,7,5
1,2,PATRICIA,JOHNSON,1,1,0
2,3,LINDA,WILLIAMS,2,4,2
3,5,ELIZABETH,BROWN,3,5,2
4,6,JENNIFER,DAVIS,3,4,1


In [16]:
# 11 Sort by biggest increase in activity
final_df.sort_values(by="difference", ascending=False).head(10)

,customer_id,first_name,last_name,may_rentals,june_rentals,difference
386,454,ALEX,GRESHAM,1,10,9
178,213,GINA,WILLIAMSON,1,9,8
248,295,DAISY,BATES,1,9,8
389,457,BILL,GAVIN,1,9,8
322,380,RUSSELL,BRINSON,1,8,7
194,234,CLAUDIA,FULLER,1,8,7
218,260,CHRISTY,VARGAS,1,8,7
481,561,IAN,STILL,2,9,7
23,27,SHIRLEY,ALLEN,1,8,7
224,267,MARGIE,WADE,3,9,6


In [17]:
###STEP 3

In [18]:
# 1 Function: rental_count_month

def rental_count_month(df, month, year):
    
    # Create dynamic column name
    column_name = f"rentals_{month:02d}_{year}"
    
    # Count rentals by customer_id
    rental_count_df = (
        df.groupby("customer_id")
        .size()
        .reset_index(name=column_name)
    )
    
    return rental_count_df

In [19]:
#2 function for May 2005
may_rental_counts = rental_count_month(
    may_rentals,
    5,
    2005
)
may_rental_counts.head()

,customer_id,rentals_05_2005
0,1,2
1,2,1
2,3,2
3,5,3
4,6,3


In [20]:
#3 function for June 2005
june_rental_counts = rental_count_month(
    june_rentals,
    6,
    2005
)
june_rental_counts.head()

,customer_id,rentals_06_2005
0,1,7
1,2,1
2,3,4
3,4,6
4,5,5


In [21]:
#4
may_rental_counts = rental_count_month(may_rentals, 5, 2005)
may_rental_counts.head()

,customer_id,rentals_05_2005
0,1,2
1,2,1
2,3,2
3,5,3
4,6,3


In [22]:
### STEP 4

In [23]:
#1 Function compare_rentals

def compare_rentals(df1, df2):
    
# Merge both DataFrames using customer_id
    comparison_df = pd.merge(
        df1,
        df2,
        on="customer_id",
        how="inner"
    )
    
# Get rental column names automatically
    rental_col_1 = comparison_df.columns[1]
    rental_col_2 = comparison_df.columns[2]
    
# Create difference column
    comparison_df["difference"] = (
        comparison_df[rental_col_2]
        - comparison_df[rental_col_1]
    )
    return comparison_df

In [24]:
#2 Compare May and June rentals

comparison = compare_rentals(
    may_rental_counts,
    june_rental_counts
)
comparison.head()

,customer_id,rentals_05_2005,rentals_06_2005,difference
0,1,2,7,5
1,2,1,1,0
2,3,2,4,2
3,5,3,5,2
4,6,3,4,1


In [25]:
#3
comparison.sort_values(by="difference", ascending=False)

,customer_id,rentals_05_2005,rentals_06_2005,difference
386,454,1,10,9
178,213,1,9,8
248,295,1,9,8
389,457,1,9,8
322,380,1,8,7
...,...,...,...,...
11,14,5,1,-4
509,596,6,2,-4
210,250,5,1,-4
136,161,6,2,-4
